In [2]:
import os
import pandas as pd

folder_path = "./csv-folder/old-csv-folder"  # Change to your folder path
output_file = "old-csv_structure.txt"

with open(output_file, "w") as f:
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            file_path = os.path.join(folder_path, filename)
            try:
                df = pd.read_csv(file_path, nrows=0)  # Only read headers
                f.write(f"File: {filename}\n")
                f.write(f"Columns: {list(df.columns)}\n")
                f.write("---------------\n")
            except Exception as e:
                f.write(f"Error reading {filename}: {e}\n")
                f.write("---------------\n")

print(f"CSV structure saved to {output_file}")



CSV structure saved to old-csv_structure.txt


In [3]:

folder_path = "./csv-folder/new-csv-folder"  # Change to your folder path
output_file = "new-csv_structure.txt"

with open(output_file, "w") as f:
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            file_path = os.path.join(folder_path, filename)
            try:
                df = pd.read_csv(file_path, nrows=0)  # Only read headers
                f.write(f"File: {filename}\n")
                f.write(f"Columns: {list(df.columns)}\n")
                f.write("---------------\n")
            except Exception as e:
                f.write(f"Error reading {filename}: {e}\n")
                f.write("---------------\n")

print(f"CSV structure saved to {output_file}")


CSV structure saved to new-csv_structure.txt


In [10]:
import pandas as pd
import os

# Folder containing your CSV files
folder_path = "./csv-folder/new-csv-folder"  # change this to your folder

# Load main report table
report_df = pd.read_csv(os.path.join(folder_path, "Report.csv"))

# Function to load schedule CSVs safely
def load_schedule(file_name):
    path = os.path.join(folder_path, file_name)
    if os.path.exists(path):
        return pd.read_csv(path)
    else:
        print(f"Warning: {file_name} not found!")
        return pd.DataFrame()

# Load all schedules
schedule_a = load_schedule("ScheduleA.csv")
schedule_b = load_schedule("ScheduleB.csv")
schedule_c = load_schedule("ScheduleC.csv")
schedule_d = load_schedule("ScheduleD.csv")
schedule_e = load_schedule("ScheduleE.csv")
schedule_f = load_schedule("ScheduleF.csv")
schedule_g = load_schedule("ScheduleG.csv")
schedule_h = load_schedule("ScheduleH.csv")
schedule_i = load_schedule("ScheduleI.csv")

# Aggregate totals by ReportId and rename to avoid duplicates
def aggregate_schedule(df, total_col_name="Amount", new_col_name=None):
    if df.empty or total_col_name not in df.columns:
        return pd.DataFrame()
    agg = df.groupby("ReportId")[total_col_name].sum().reset_index()
    if new_col_name:
        agg = agg.rename(columns={total_col_name: new_col_name})
    return agg

# Create aggregated totals for schedules
a_totals = aggregate_schedule(schedule_a, "Amount", "ScheduleA_Total")
b_totals = aggregate_schedule(schedule_b, "Amount", "ScheduleB_Total")
c_totals = aggregate_schedule(schedule_c, "Amount", "ScheduleC_Total")
d_totals = aggregate_schedule(schedule_d, "Amount", "ScheduleD_Total")
e_totals = aggregate_schedule(schedule_e, "Amount", "ScheduleE_Total")
f_totals = aggregate_schedule(schedule_f, "Amount", "ScheduleF_Total")
i_totals = aggregate_schedule(schedule_i, "Amount", "ScheduleI_Total")

# Merge totals into report_df
merged_df = report_df.copy()
for df in [a_totals, b_totals, c_totals, d_totals, e_totals, f_totals, i_totals]:
    if not df.empty:
        merged_df = pd.merge(merged_df, df, on="ReportId", how="left")

# Merge ScheduleG totals (select only columns with 'Total' or 'Count')
if not schedule_g.empty:
    g_cols = ["ReportId"] + [c for c in schedule_g.columns if "Total" in c or "Count" in c]
    merged_df = pd.merge(merged_df, schedule_g[g_cols], on="ReportId", how="left")

# Merge ScheduleH totals/balances (select columns with 'Total', 'Balance', or 'Funds')
if not schedule_h.empty:
    h_cols = ["ReportId"] + [c for c in schedule_h.columns if any(x in c for x in ["Total", "Balance", "Funds"])]
    merged_df = pd.merge(merged_df, schedule_h[h_cols], on="ReportId", how="left")

# Save the merged dataset
output_file = os.path.join(folder_path, "Merged_VA_Campaign_Finance.csv")
merged_df.to_csv(output_file, index=False)
print(f"Merged dataset saved to {output_file}")


Merged dataset saved to ./csv-folder/new-csv-folder/Merged_VA_Campaign_Finance.csv


In [22]:
import pandas as pd
from pathlib import Path

# -----------------------
# Base directory
# -----------------------
base = Path("./csv-folder/new-csv-folder")

# -----------------------
# Load main report table
# -----------------------
report = pd.read_csv(base / "Report.csv")

# Normalize CandidateName and CommitteeName
report["CandidateNameNorm"] = report["CandidateName"].str.strip().str.lower()
report["CommitteeNameNorm"] = report["CommitteeName"].str.strip().str.lower()

# -----------------------
# Define transactional schedules only
# -----------------------
transactional_schedules = ["ScheduleA.csv", "ScheduleB.csv", "ScheduleC.csv",
                           "ScheduleD.csv", "ScheduleF.csv", "ScheduleI.csv"]

frames = []

for file_name in transactional_schedules:
    file_path = base / file_name
    if not file_path.exists():
        print(f"Skipping {file_name}, file not found")
        continue
    
    df = pd.read_csv(file_path)
    
    # Determine which column is the contribution/expenditure amount
    possible_amount_cols = ["Amount", "TotalToDate"]
    amount_col = next((c for c in possible_amount_cols if c in df.columns), None)
    
    if amount_col is None:
        print(f"Skipping {file_name}, no Amount column found")
        continue
    
    # Keep relevant columns
    cols = [c for c in ["ReportId", amount_col, "LastOrCompanyName", "TransactionDate"] if c in df.columns]
    df = df[cols].copy()
    
    # Rename amount column to "Amount"
    df = df.rename(columns={amount_col: "Amount"})
    
    # Merge CandidateName from Report.csv
    df = df.merge(report[["ReportId","CandidateName","CommitteeName","ReportYear","Party","OfficeSought","District"]],
                  on="ReportId", how="left")
    
    # Fill missing Amounts
    df["Amount"] = df["Amount"].fillna(0)
    
    # Normalize CandidateName
    df["CandidateName"] = df["CandidateName"].str.strip().str.lower()
    
    frames.append(df)

# Combine all transactional schedules
merged = pd.concat(frames, ignore_index=True)

# -----------------------
# Filter out rows without a candidate
# -----------------------
merged = merged[merged["CandidateName"].notna() & (merged["CandidateName"] != "")]

# -----------------------
# Helper function to aggregate
# -----------------------
valid_group_cols = ["CommitteeName","CandidateName","Party","ReportYear",
                    "OfficeSought","District","LastOrCompanyName","TransactionDate"]

def sum_by(group_cols, top_n=None, year=None, filters=None):
    df_filtered = merged.copy()
    
    # Apply filters
    if filters:
        for col, keyword in filters.items():
            if col in df_filtered.columns:
                df_filtered = df_filtered[df_filtered[col].str.contains(keyword.lower(), na=False)]
    
    # Apply year filter
    if year and "ReportYear" in df_filtered.columns:
        df_filtered = df_filtered[df_filtered["ReportYear"] == year]
    
    # Validate grouping columns
    bad = [c for c in group_cols if c not in valid_group_cols]
    if bad:
        raise ValueError(f"Invalid group cols: {bad}")
    
    # Group and sum
    result = df_filtered.groupby(group_cols, dropna=False)["Amount"]\
                        .sum().reset_index().sort_values("Amount", ascending=False)
    
    if top_n:
        result = result.head(top_n)
    
    return result

# -----------------------
# Save merged table
# -----------------------
merged.to_csv(base / "newmerged_va_campaign_finance.csv", index=False)
print(f"Merged file saved to {base / 'newmerged_va_campaign_finance.csv'}")



Merged file saved to csv-folder/new-csv-folder/newmerged_va_campaign_finance.csv


In [32]:
import pandas as pd
from pathlib import Path

# -----------------------
# Base directory
# -----------------------
base = Path("./csv-folder/new-csv-folder")

# -----------------------
# Load main report table
# -----------------------
report = pd.read_csv(base / "Report.csv")

# Normalize candidate and committee names
report["CandidateNameNorm"] = report["CandidateName"].str.strip().str.lower()
report["CommitteeNameNorm"] = report["CommitteeName"].str.strip().str.lower()

# -----------------------
# Transactional schedules only
# -----------------------
transactional_schedules = ["ScheduleA.csv", "ScheduleB.csv", "ScheduleC.csv",
                           "ScheduleD.csv", "ScheduleF.csv", "ScheduleI.csv"]

frames = []

for file_name in transactional_schedules:
    file_path = base / file_name
    if not file_path.exists():
        print(f"Skipping {file_name}, file not found")
        continue
    
    df = pd.read_csv(file_path)
    
    # Detect numeric contribution/expenditure column
    possible_amount_cols = ["Amount", "TotalToDate"]
    amount_col = next((c for c in possible_amount_cols if c in df.columns), None)
    
    if amount_col is None:
        print(f"Skipping {file_name}, no Amount column found")
        continue
    
    # Keep relevant columns
    cols = [c for c in ["ReportId", amount_col, "LastOrCompanyName", "TransactionDate"] if c in df.columns]
    df = df[cols].copy()
    
    # Rename amount column to "Amount"
    df = df.rename(columns={amount_col: "Amount"})
    
    # Merge candidate and committee info
    df = df.merge(
        report[["ReportId","CandidateName","CommitteeName","ReportYear","Party","OfficeSought","District"]],
        on="ReportId", how="left"
    )
    
    # Fill missing Amounts
    df["Amount"] = df["Amount"].fillna(0)
    
    # Normalize strings for filtering
    df["CandidateName"] = df["CandidateName"].str.strip().str.lower()
    df["LastOrCompanyNameNorm"] = df["LastOrCompanyName"].str.strip().str.lower()
    
    frames.append(df)

# Combine all transactional schedules
merged = pd.concat(frames, ignore_index=True)

# Filter out rows without a candidate
merged = merged[merged["CandidateName"].notna() & (merged["CandidateName"] != "")]

# -----------------------
# Helper function to aggregate
# -----------------------
valid_group_cols = ["CommitteeName","CandidateName","Party","ReportYear",
                    "OfficeSought","District","LastOrCompanyName","TransactionDate"]

def sum_by(group_cols, top_n=None, year=None, filters=None):
    df_filtered = merged.copy()
    
    # Apply filters (case-insensitive)
    if filters:
        for col, keyword in filters.items():
            if col in df_filtered.columns:
                # Use normalized column if it exists
                norm_col = col + "Norm" if col + "Norm" in df_filtered.columns else col
                df_filtered = df_filtered[df_filtered[norm_col].str.contains(keyword.lower(), na=False)]
    
    # Apply year filter
    if year and "ReportYear" in df_filtered.columns:
        df_filtered = df_filtered[df_filtered["ReportYear"] == year]
    
    # Validate grouping columns
    bad = [c for c in group_cols if c not in valid_group_cols]
    if bad:
        raise ValueError(f"Invalid group cols: {bad}")
    
    # Group and sum
    result = df_filtered.groupby(group_cols, dropna=False)["Amount"]\
                        .sum().reset_index().sort_values("Amount", ascending=False)
    
    if top_n:
        result = result.head(top_n)
    
    return result

# -----------------------
# Save merged table
# -----------------------
merged.to_csv(base / "merged_va_campaign_finance.csv", index=False)
print(f"Merged file saved to {base / 'merged_va_campaign_finance.csv'}")

# -----------------------
# Example queries
# -----------------------
# Top 10 committees by expenditures
top_committees = sum_by(["CommitteeName","ReportYear"], top_n=10)
print(top_committees.head())

# Dominion contributions by candidate (case-insensitive)
dominion_by_candidate = sum_by(["CandidateName"], filters={"LastOrCompanyName":"dominion"})
print(dominion_by_candidate.head())


Merged file saved to csv-folder/new-csv-folder/merged_va_campaign_finance.csv
                             CommitteeName  ReportYear     Amount
18                   Leftwich for Delegate      2023.0  315503.11
12  Friends of Rachel Levy - 59th District      2023.0  138428.75
4                       Culipher for Clerk      2024.0   49262.90
26                    VoteSandySchoolBoard      2023.0   38074.94
5              Friends of Erika Guess 2022      2022.0   27176.47
                 CandidateName   Amount
0  james jay" a. leftwich jr."  20000.0


In [33]:

# Dominion contributions by candidate (case-insensitive)
dominion_by_candidate = sum_by(["CandidateName"], filters={"LastOrCompanyName":"dominion"})
print(dominion_by_candidate.head())


                 CandidateName   Amount
0  james jay" a. leftwich jr."  20000.0


In [39]:
# Cell 2: Imports and settings
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from pathlib import Path
import time
import requests
import random
import re



def init_driver():
    options = webdriver.ChromeOptions()
    if HEADLESS:
        options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    return driver

def download_file(url, save_path):
    """Download a real CSV file using requests"""
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/139.0.0.0 Safari/537.36",
        "Referer": BASE_URL
    }
    try:
        r = requests.get(url, headers=headers, timeout=60)
        r.raise_for_status()
        content = r.content
        if len(content) < 100 and (b'error' in content.lower() or b'forbidden' in content.lower() or b'<html' in content.lower()):
            print(f"   ❌ {save_path.name} appears to be an error page, skipping")
            return False
        save_path.write_bytes(content)
        print(f"   ✅ Downloaded {save_path.name} ({len(content):,} bytes)")
        return True
    except Exception as e:
        print(f"   ❌ Failed {save_path.name}: {e}")
        return False

# === Settings ===
BASE_URL = "https://apps.elections.virginia.gov/SBE_CSV/CF/"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
HEADLESS = True  # Set False to watch browser
driver = init_driver()
driver.get(BASE_URL)
html = driver.page_source
print(html[:1000])  # print first 1000 characters to see what the page looks like


<html><head>
<title>Access Denied</title>
</head><body>
<h1>Access Denied</h1>
 
You don't have permission to access "http://apps.elections.virginia.gov/SBE_CSV/CF/" on this server.<p>
Reference #18.8809c617.1756158632.a30aff
</p><p>https://errors.edgesuite.net/18.8809c617.1756158632.a30aff</p>


</body></html>
